In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')


In [ ]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"] = 0
true["label"] = 1

data = pd.concat([fake, true])
data = data.sample(frac=1).reset_index(drop=True)

data.head()


In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

data["content"] = data["title"] + " " + data["text"]
data["clean_text"] = data["content"].apply(clean_text)



In [ ]:
X = data["clean_text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [ ]:
model = LogisticRegression()
model.fit(X_train_vec, y_train)


In [ ]:
pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))


In [ ]:
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
def predict_news(news_text):
    news_text = clean_text(news_text)
    vector = vectorizer.transform([news_text])
    prediction = model.predict(vector)

    if prediction[0] == 1:
        return "🟢 This looks like REAL NEWS"
    else:
        return "🔴 This looks like FAKE / MISINFORMATION"


In [ ]:
print("AI Fake News Detector")
print("----------------------")

while True:
    user_news = input("\nEnter a news headline or full news script (or type 'exit'): ")

    if user_news.lower() == "exit":
        print("Program ended.")
        break

    result = predict_news(user_news)
    print("\nPrediction:", result)
